# Vibration Isolation — As-Selected Re-Solve (COTS FC)
### Electric V-BAT-Like Tail-Sitter (EDF) — Conceptual Design Study

---

## Purpose

NB5 (`vibration_isolation`) soft-mounted a *configured estimate* of the
FC/IMU cluster (`config/vibration.yaml`: 60 g). NB11 froze a real flight
controller. This notebook **re-solves the isolator sizing with the frozen
FC's catalog mass** — downstream, as a new notebook, keeping the pipeline
one-way (ADR-0012).

Because the 1-DOF isolator corner frequency depends only on the forcing
frequency, target, and damping — *not* on the isolated mass — the corner
frequency, transmissibility, static sag, and sway space are unchanged by
construction; what the real FC mass changes is the **isolator stiffness**
(k ∝ m) the parts must be procured to. The forcing side is also
re-anchored: the blade count now comes with a consistency check against
the frozen propeller's rating.

## Inputs

- `out/components.yaml` *(NB11 — frozen FC and propeller)*
- `out/vibration.yaml` *(NB5 — the conceptual solution, for the delta report)*
- `config/vibration.yaml`, `config/prop_geometry.yaml`, rotor RPM law

## Outputs

- `out/vibration_cots.yaml` *(same schema as `out/vibration.yaml` + frozen-FC
  block; consumed by NB14)*

---

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # bare-checkout fallback

from dataclasses import replace

import yaml

from conceptual_design.notebook import nb_setup
from conceptual_design.design_point import load_handoff, solve_design_point
from conceptual_design import FrozenComponents, reports
from conceptual_design.vtol_power import VTOLParams, rpm_from_diameter
from conceptual_design.vibration_isolation import (
    VibrationParams, size_isolation, write_vibration_yaml,
)

nb = nb_setup(style=False)   # no figures in this notebook
REPO_ROOT, CONFIG_PATH, OUT_PATH, FIG_DIR = (
    nb.repo_root, nb.config, nb.out, nb.fig_dir)

# Section 1 — Design Inputs

Re-run the sizing loop from `config/` (same pattern as NB2–NB12), derive
the EDF forcing frequency, and load the frozen hardware. The configured
blade count must match the frozen propeller — a mismatch means
`config/prop_geometry.yaml` and the COTS freeze have drifted apart.

---

In [ ]:
# -- Re-run the sizing loop + read the freeze and the NB5 estimate -------------
inp, result = solve_design_point(CONFIG_PATH)
vib_p = VibrationParams.from_yaml(CONFIG_PATH / "vibration.yaml")

with open(CONFIG_PATH / "prop_geometry.yaml") as f:
    n_blades = int(yaml.safe_load(f)["n_blades"])

vib_est = load_handoff(OUT_PATH, "vibration")
comps   = FrozenComponents.from_yaml(OUT_PATH / "components.yaml")
fc      = comps["flight_controller"]
prop_frozen = comps["propeller"]

assert int(prop_frozen.ratings["n_blades"]) == n_blades, (
    f"config/prop_geometry.yaml n_blades={n_blades} != frozen propeller "
    f"{prop_frozen.id} ({prop_frozen.ratings['n_blades']} blades) -- "
    "the configured rotor geometry has drifted from the COTS freeze")

vtol = VTOLParams.from_propulsive(inp.prop, inp.env)
rpm  = rpm_from_diameter(inp.rotor.D_rotor_m, vtol)

print(f"EDF shaft speed   : {rpm:.0f} RPM  -> {rpm/60:.1f} Hz (1/rev)")
print(f"Blade count       : {n_blades}  (matches frozen {prop_frozen.id})")
print(f"Frozen FC         : {fc.name}  ({fc.id})")
print(f"  mass             : {fc.mass_kg*1e3:.1f} g   (NB5 estimate: {vib_p.m_fc_imu_kg*1e3:.1f} g)")
print(f"Payload (isolated): {result.m_payload_kg:.3f} kg  (unchanged -- not a COTS selection)")

# Section 2 — Re-Solve with the Frozen FC Mass

Same physics as NB5 (`size_isolation`, one call), with the configured
FC/IMU cluster estimate replaced by the frozen FC's catalog mass. Corner
frequency and window come out identical by construction; the FC-tray
isolator stiffness scales down with the lighter real board.

---

In [ ]:
# -- Re-solve NB5 with the frozen FC mass --------------------------------------
vib_p_cots = replace(vib_p, m_fc_imu_kg=fc.mass_kg)
res = size_isolation(rpm, n_blades, result.m_payload_kg, vib_p_cots)

lo, hi = res["f_n_window_hz"]
print(f"Forcing (1/rev)   : {res['f_shaft_hz']:.1f} Hz")
print(f"Corner freq f_n   : {res['f_n_hz']:.1f} Hz   "
      f"(valid window {lo:.0f}-{hi:.0f} Hz -> {'OK' if res['window_ok'] else 'OUT OF WINDOW'})")
print(f"Sway per bay      : {res['sway_mm']:.2f} mm  (unchanged -- mass-independent)")
print()
reports.print_vibration_cots_table(res)

k_fc_est  = vib_est["modules"]["fc_imu"]["k_isolator_N_m"]
k_fc_cots = res["modules"]["fc_imu"].k_isolator_N_m
print()
print(f"FC-tray isolator stiffness: {k_fc_est:.0f} -> {k_fc_cots:.0f} N/m "
      f"({(k_fc_cots/k_fc_est - 1)*100:+.0f}% -- procure to this value)")

assert res["window_ok"], "corner frequency outside the valid isolation window"

# Section 3 — Output Export

`out/vibration_cots.yaml` — same schema as `out/vibration.yaml` (so the
fuselage re-solve consumes it identically) plus a `cots_fc` provenance
block.

---

In [ ]:
write_vibration_yaml(
    res, vib_p_cots, OUT_PATH / "vibration_cots.yaml",
    regen_notebook="notebooks/vibration_isolation_cots.ipynb",
    extra={"cots_fc": {
        "id":     fc.id,
        "name":   fc.name,
        "mass_g": round(fc.mass_kg * 1e3, 2),
    }},
)
print(f"As-selected vibration design written -> {OUT_PATH / 'vibration_cots.yaml'}")

# Section 4 — Delta vs. the Conceptual Solution (NB5) and Summary

---

In [ ]:
reports.print_vibration_cots_summary(res, vib_est, fc)